# Notebook 3 - Feature Engineering

This notebook turns the cleaned outputs from **Notebook 1** into a modeling-ready feature table.

The main goal here is to build **pre-race historical features** for each Laurel Park horse-race entry, while carefully avoiding leakage from the current target race.

We will build three layers:

1. **Target/base table**  
   one row per Laurel horse-race with the outcome columns and target-race context

2. **Traditional PP features**  
   engineered only from the horse's recent **historical traditional past performances**

3. **Historical GPS features**  
   engineered only from the horse's recent **historical GPS past performances**

At the end, we save:

- `target_base_table.csv`
- `historical_pp_selected.csv`
- `traditional_feature_table.csv`
- `gps_pp_race_summary.csv`
- `historical_gps_selected.csv`
- `gps_feature_table.csv`
- `modeling_feature_table.csv`
- `feature_inventory.csv`

## What features this notebook builds

### Target/base context features
These are not "historical" features, but they are useful context columns to keep in the final modeling table:

- `target_distance_furlongs`
- `target_surface`
- `target_race_type`
- `target_purse`
- `field_size`
- `post_position`

### Traditional historical features
We build both **summary features** and **last-3-race sequence features**.

**Summary features**
- number of prior starts
- days since last start
- mean / best / worst / std of finish position
- win rate and top-3 rate
- mean post-time odds
- mean field size and purse
- share of starts on the same surface as the target race
- mean absolute distance gap to the target race
- share of prior starts that had GPS available
- distance-aware point-of-call style features:
  - early-call position
  - late-call position
  - early-to-late gain
  - early-to-finish gain
  - early / late / finish margin behind
  - margin recovery

**Sequence features for the last 3 starts**
For each of `last`, `back2`, and `back3`, we keep:
- race date, track, race number
- surface, race type, distance label
- days since the target race
- distance gap to target
- same-surface flag
- finish position, field size, post position, odds, purse
- number of active point-of-calls
- early / mid / late call position
- early / late margin behind
- early-to-late gain
- early-to-finish gain
- length-behind recovery

### Historical GPS features
Again, we build both **summary features** and **last-3-GPS-race sequence features**.

**GPS summary features**
- number of historical GPS starts
- days since last GPS start
- mean / best / std of finish position
- win rate and top-3 rate
- same-surface share
- distance-gap summary
- mean number of GPS gate rows
- early / late position
- early-to-late position gain
- early / late distance behind
- distance-behind recovery
- early / late sectional-time behavior
- path efficiency ratio
- extra distance run versus expected distance
- meters per stride
- final running time

**GPS sequence features for the last 3 GPS starts**
For each of `last`, `back2`, and `back3`, we keep:
- race date, track, race number
- surface, race type, distance label
- days since target race
- distance gap to target
- same-surface flag
- finish position, field size, post position, purse
- number of gate rows
- early / late / best / worst / mean position
- position-gain profile
- sectional-time profile
- distance-behind profile
- final running time
- cumulative distance / strides
- path efficiency ratio
- extra distance
- meters per stride

This keeps Notebook 3 rich enough for modeling later, but still organized.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 180)

## 1. Data Loading

This notebook expects the CSVs written by Notebook 1 to exist in `data/processed/`. If they are missing, run Notebook 1 first.

In [45]:
PROCESSED_DIR_CANDIDATES = [
    Path("../data/processed"),
]

PROCESSED_DIR = None
for p in PROCESSED_DIR_CANDIDATES:
    if p.exists():
        PROCESSED_DIR = p
        break

if PROCESSED_DIR is None:
    raise FileNotFoundError(
        "Could not find data/processed. Please run Notebook 1 first so the cleaned CSVs are created."
    )

FEATURE_DIR = Path("../data/features")
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

required_files = {
    "laurel_flat": "laurel_horse_race_flat.csv",
    "pps_clean": "pps_clean.csv",
    "gps_pp_clean": "gps_pp_clean.csv",
    "poc_profile": "point_of_call_profile_by_distance.csv",
}

missing_files = [name for name, fname in required_files.items() if not (PROCESSED_DIR / fname).exists()]
if missing_files:
    raise FileNotFoundError(
        f"Missing Notebook 1 outputs for: {missing_files}. Found processed dir at {PROCESSED_DIR}."
    )

laurel_flat = pd.read_csv(PROCESSED_DIR / required_files["laurel_flat"], parse_dates=["race_date"])
pps_clean = pd.read_csv(PROCESSED_DIR / required_files["pps_clean"], parse_dates=["pp_race_date"])
gps_pp_clean = pd.read_csv(PROCESSED_DIR / required_files["gps_pp_clean"], parse_dates=["pp_race_date"])
point_of_call_profile_by_distance = pd.read_csv(PROCESSED_DIR / required_files["poc_profile"])

inventory = pd.DataFrame(
    {
        "table": ["laurel_flat", "pps_clean", "gps_pp_clean", "point_of_call_profile_by_distance"],
        "rows": [len(laurel_flat), len(pps_clean), len(gps_pp_clean), len(point_of_call_profile_by_distance)],
        "columns": [
            laurel_flat.shape[1],
            pps_clean.shape[1],
            gps_pp_clean.shape[1],
            point_of_call_profile_by_distance.shape[1],
        ],
    }
)
display(inventory)

/var/folders/10/yh5g9ksn5fb_2pzdwtzd2tlh0000gn/T/ipykernel_89883/3224705395.py:34: DtypeWarning: Columns (1,9) have mixed types. Specify dtype option on import or set low_memory=False.
  gps_pp_clean = pd.read_csv(PROCESSED_DIR / required_files["gps_pp_clean"], parse_dates=["pp_race_date"])


,table,rows,columns
0,laurel_flat,1054,186
1,pps_clean,2293,44
2,gps_pp_clean,23971,33
3,point_of_call_profile_by_distance,18,11


## 2. Helper functions

These helpers do three main jobs:

1. create a clean target/base table
2. convert variable point-of-call layouts into generic early/mid/late features
3. summarize a GPS gate-level prior race into one race-level record

In [3]:
def maybe_to_numeric(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    out = df.copy()
    for col in columns:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out

In [4]:
def first_valid(values):
    for v in values:
        if pd.notna(v):
            return v
    return np.nan

In [5]:
def second_valid(values):
    seen = 0
    for v in values:
        if pd.notna(v):
            seen += 1
            if seen == 2:
                return v
    return np.nan

In [ ]:
def last_valid(values):
    for v in values[::-1]:
        if pd.notna(v):
            return v
    return np.nan

In [7]:
def middle_valid(values):
    valid = [v for v in values if pd.notna(v)]
    if not valid:
        return np.nan
    return valid[len(valid) // 2]

In [8]:
def add_generic_point_of_call_features(df: pd.DataFrame) -> pd.DataFrame:
    '''
    Convert the raw point-of-call columns into generic early/mid/late features.

    This is the easiest way to respect the fact that different race distances
    have different point-of-call layouts.
    '''
    out = df.copy()

    position_cols = [f"position_at_point_of_call_{i}" for i in range(1, 6)]
    behind_cols = [f"length_behind_at_poc_{i}" for i in range(1, 6)]
    ahead_cols = [f"length_ahead_at_poc_{i}" for i in range(1, 6)]

    position_cols = [c for c in position_cols if c in out.columns]
    behind_cols = [c for c in behind_cols if c in out.columns]
    ahead_cols = [c for c in ahead_cols if c in out.columns]

    pos_vals = out[position_cols].to_numpy(dtype=float)
    behind_vals = out[behind_cols].to_numpy(dtype=float) / 100.0
    ahead_vals = out[ahead_cols].to_numpy(dtype=float) / 100.0

    out["n_active_point_of_calls"] = np.sum(~np.isnan(pos_vals), axis=1)
    out["early_call_position"] = [first_valid(list(r)) for r in pos_vals]
    out["second_call_position"] = [second_valid(list(r)) for r in pos_vals]
    out["mid_call_position"] = [middle_valid(list(r)) for r in pos_vals]
    out["late_call_position"] = [last_valid(list(r)) for r in pos_vals]

    out["early_call_length_behind"] = [first_valid(list(r)) for r in behind_vals]
    out["second_call_length_behind"] = [second_valid(list(r)) for r in behind_vals]
    out["mid_call_length_behind"] = [middle_valid(list(r)) for r in behind_vals]
    out["late_call_length_behind"] = [last_valid(list(r)) for r in behind_vals]

    out["early_call_length_ahead"] = [first_valid(list(r)) for r in ahead_vals]
    out["late_call_length_ahead"] = [last_valid(list(r)) for r in ahead_vals]

    if "length_behind_at_finish" in out.columns:
        out["finish_length_behind"] = pd.to_numeric(out["length_behind_at_finish"], errors="coerce") / 100.0
    if "length_ahead_at_finish" in out.columns:
        out["finish_length_ahead"] = pd.to_numeric(out["length_ahead_at_finish"], errors="coerce") / 100.0

    out["early_to_late_call_gain"] = out["early_call_position"] - out["late_call_position"]
    out["early_to_finish_gain"] = out["early_call_position"] - out["official_position"]
    out["late_call_to_finish_gain"] = out["late_call_position"] - out["official_position"]
    out["length_behind_recovery"] = out["early_call_length_behind"] - out["finish_length_behind"]

    return out

In [9]:
def summarize_single_gps_race(g: pd.DataFrame) -> pd.Series:
    '''
    Summarize one historical GPS PP race from gate-level rows to one horse-race row.
    '''
    g = g.sort_values("gate", ascending=False).copy()  # earliest gate to latest gate
    n = len(g)

    if n == 0:
        return pd.Series(dtype="object")

    early_n = max(int(np.ceil(n / 3)), 1)
    late_n = max(int(np.ceil(n / 3)), 1)

    early = g.iloc[:early_n]
    late = g.iloc[-late_n:]
    latest = g.sort_values("gate", ascending=True).iloc[0]  # gate 0 should usually be here

    distance_expected = pd.to_numeric(g["distance_meters_expected"].iloc[0], errors="coerce")
    cumulative_distance_final = pd.to_numeric(latest["cumulative_distance_ran"], errors="coerce")
    cumulative_strides_final = pd.to_numeric(latest["cumulative_strides"], errors="coerce")

    if pd.notna(cumulative_distance_final) and pd.notna(distance_expected) and distance_expected != 0:
        distance_efficiency_ratio = cumulative_distance_final / distance_expected
        extra_distance_m = cumulative_distance_final - distance_expected
    else:
        distance_efficiency_ratio = np.nan
        extra_distance_m = np.nan

    if pd.notna(cumulative_distance_final) and pd.notna(cumulative_strides_final) and cumulative_strides_final != 0:
        meters_per_stride = cumulative_distance_final / cumulative_strides_final
    else:
        meters_per_stride = np.nan

    return pd.Series(
        {
            "horse_name": g["horse_name"].iloc[0],
            "registration_number": g["registration_number"].iloc[0],
            "pp_track": g["pp_track"].iloc[0],
            "pp_race_date": g["pp_race_date"].iloc[0],
            "pp_race_number": g["pp_race_number"].iloc[0],
            "pp_race_key": g["pp_race_key"].iloc[0],
            "surface": g["surface"].iloc[0],
            "race_type": g["race_type"].iloc[0],
            "distance_label": g["distance_label"].iloc[0],
            "distance_furlongs": g["distance_furlongs"].iloc[0],
            "distance_meters_expected": distance_expected,
            "purse": g["purse"].iloc[0],
            "field_size": g["field_size"].iloc[0],
            "post_position": g["post_position"].iloc[0],
            "official_position": g["official_position"].iloc[0],
            "num_gate_rows": n,
            "early_position": early["position"].mean(),
            "late_position": late["position"].mean(),
            "best_position": g["position"].min(),
            "worst_position": g["position"].max(),
            "mean_position": g["position"].mean(),
            "std_position": g["position"].std(),
            "position_gain_early_to_late": early["position"].mean() - late["position"].mean(),
            "sectional_time_mean": g["sectional_time"].mean(),
            "sectional_time_std": g["sectional_time"].std(),
            "sectional_time_early_mean": early["sectional_time"].mean(),
            "sectional_time_late_mean": late["sectional_time"].mean(),
            "late_minus_early_sectional": late["sectional_time"].mean() - early["sectional_time"].mean(),
            "distance_behind_early": early["distance_behind"].mean(),
            "distance_behind_late": late["distance_behind"].mean(),
            "distance_behind_mean": g["distance_behind"].mean(),
            "distance_behind_max": g["distance_behind"].max(),
            "distance_behind_recovery": early["distance_behind"].mean() - late["distance_behind"].mean(),
            "time_behind_mean": g["time_behind"].mean(),
            "final_running_time": latest["running_time"],
            "cumulative_distance_final": cumulative_distance_final,
            "cumulative_strides_final": cumulative_strides_final,
            "distance_efficiency_ratio": distance_efficiency_ratio,
            "extra_distance_m": extra_distance_m,
            "meters_per_stride": meters_per_stride,
        }
    )

## 3. Build the Laurel target/base table

This is the table that represents the **current Laurel Park race entries**.

Important:
- this is **one row per horse-race**
- we keep the outcome columns here
- we also create simple target variables that later notebooks can use:
  - `finish_percentile`
  - `is_winner`
  - `is_top_3`

These target variables are okay to build here because this notebook is constructing the final modeling table layout, not fitting a model yet.

### Why keep these target/base context columns?

These columns are not historical performance features, but they matter because race outcomes are heavily shaped by **race conditions**. A horse running today at a different distance, surface, or class level is not competing in the same environment as in a prior start.

I keep these columns for three reasons:

1. **Context for interpretation.** If a horse's historical record looks strong, that strength still needs to be interpreted relative to today's race setup.
2. **Better comparisons across horses.** Two horses with similar past results may have very different prospects if one is stretching out in distance, switching surface, or moving into a larger field.
3. **Cleaner feature engineering later.** Several historical features are defined relative to the target race, such as distance gap and same-surface indicators, so keeping the target-race context explicitly in the base table makes those comparisons straightforward and reproducible.

The target variables are also built here because this notebook is organizing the modeling frame, but they are not used to construct the historical features themselves, which helps avoid leakage.


In [11]:
target_base = laurel_flat.copy()

target_base = maybe_to_numeric(
    target_base,
    [
        "distance_furlongs",
        "purse",
        "field_size",
        "post_position",
        "official_position",
    ],
)

target_base["finish_percentile"] = np.where(
    target_base["field_size"] > 1,
    (target_base["official_position"] - 1) / (target_base["field_size"] - 1),
    np.nan,
)
target_base["is_winner"] = (target_base["official_position"] == 1).astype("Int64")
target_base["is_top_3"] = (target_base["official_position"] <= 3).astype("Int64")

target_base["target_race_date"] = pd.to_datetime(target_base["race_date"])
target_base["target_distance_furlongs"] = pd.to_numeric(target_base["distance_furlongs"], errors="coerce")
target_base["target_surface"] = target_base["surface"].astype("string")
target_base["target_race_type"] = target_base["race_type"].astype("string")
target_base["target_purse"] = pd.to_numeric(target_base["purse"], errors="coerce")

target_keep_cols = [
    "horse_race_key",
    "race_key",
    "registration_number",
    "horse_name",
    "target_race_date",
    "race_number",
    "target_distance_furlongs",
    "distance_label",
    "target_surface",
    "target_race_type",
    "target_purse",
    "field_size",
    "post_position",
    "official_position",
    "finish_percentile",
    "is_winner",
    "is_top_3",
]
target_base = target_base[target_keep_cols].copy()

display(target_base.head(10))

,horse_race_key,race_key,registration_number,horse_name,target_race_date,race_number,target_distance_furlongs,distance_label,target_surface,target_race_type,target_purse,field_size,post_position,official_position,finish_percentile,is_winner,is_top_3
0,LRL|2025-12-12|1|22004645,LRL|2025-12-12|1,22004645,Bourbon N Lace,2025-12-12,1,7.0,7F,D,SOC,27900,8,7,1,0.000000,1,1
1,LRL|2025-12-12|1|22010064,LRL|2025-12-12|1,22010064,Buckin' Right,2025-12-12,1,7.0,7F,D,SOC,27900,8,1,2,0.142857,0,1
2,LRL|2025-12-12|1|22002156,LRL|2025-12-12|1,22002156,Mun Mun Can Run,2025-12-12,1,7.0,7F,D,SOC,27900,8,6,3,0.285714,0,1
3,LRL|2025-12-12|1|22011846,LRL|2025-12-12|1,22011846,Rehoboth Avenue,2025-12-12,1,7.0,7F,D,SOC,27900,8,2,4,0.428571,0,0
4,LRL|2025-12-12|1|21003278,LRL|2025-12-12|1,21003278,Weekend Wife,2025-12-12,1,7.0,7F,D,SOC,27900,8,8,5,0.571429,0,0
5,LRL|2025-12-12|1|22004844,LRL|2025-12-12|1,22004844,Over My Cents,2025-12-12,1,7.0,7F,D,SOC,27900,8,5,6,0.714286,0,0
6,LRL|2025-12-12|1|19002714,LRL|2025-12-12|1,19002714,Mischief Motion,2025-12-12,1,7.0,7F,D,SOC,27900,8,4,7,0.857143,0,0
7,LRL|2025-12-12|1|21017739,LRL|2025-12-12|1,21017739,Sippin' Time,2025-12-12,1,7.0,7F,D,SOC,27900,8,3,8,1.000000,0,0
8,LRL|2025-12-12|2|20016574,LRL|2025-12-12|2,20016574,Feeling Woozy,2025-12-12,2,8.5,1 1/16M,D,SOC,40885,5,2,1,0.000000,1,1
9,LRL|2025-12-12|2|19010909,LRL|2025-12-12|2,19010909,Tops the Chart,2025-12-12,2,8.5,1 1/16M,D,SOC,40885,5,4,2,0.250000,0,1


In [12]:
display(
    pd.DataFrame(
        {
            "rows": [len(target_base)],
            "unique_horse_races": [target_base["horse_race_key"].nunique()],
            "unique_races": [target_base["race_key"].nunique()],
            "horses_with_no_prior_history_yet": [0],  # placeholder until mapping below
        }
    )
)

,rows,unique_horse_races,unique_races,horses_with_no_prior_history_yet
0,1054,1054,148,0


## 4. Traditional PP feature engineering

We first add distance-aware generic point-of-call features to the cleaned PP table.

Then we map prior PPs onto each Laurel target horse-race using:

- same `registration_number`
- `pp_race_date < target_race_date`
- keep only the **three most recent** historical starts

This is the right place to connect the tables, because this is where we are explicitly building **historical features**.

### Why use only prior races and only the three most recent starts?

This choice is doing two things at once.

First, restricting to `pp_race_date < target_race_date` enforces the real forecasting setup: for each Laurel entry, we only use information that would have been available **before** that race was run.

Second, keeping the **three most recent** starts is a practical bias toward relevance. In racing, form can change quickly because of fitness, layoffs, class moves, surface switches, and short-term improvement or decline. Older starts may still contain signal, but they are often less representative of current form.

Using the last three starts gives a reasonable balance:

- enough history to see a pattern instead of a one-race fluke,
- but not so much history that the features become stale,
- and not so many columns that the sequence table becomes sparse or overly wide for a small practice-round dataset.

This also makes the traditional and GPS pipelines comparable, since both are aligned around the same recent-history window.


In [13]:
pps_fe = add_generic_point_of_call_features(pps_clean)

pp_map = target_base[
    ["horse_race_key", "registration_number", "target_race_date", "target_distance_furlongs", "target_surface"]
].merge(
    pps_fe,
    on="registration_number",
    how="left",
)

pp_map = pp_map.loc[pp_map["pp_race_date"] < pp_map["target_race_date"]].copy()
pp_map = pp_map.sort_values(
    ["horse_race_key", "pp_race_date", "pp_race_number"],
    ascending=[True, False, False],
).reset_index(drop=True)

pp_map["pp_rank_desc"] = pp_map.groupby("horse_race_key").cumcount() + 1
pp_map = pp_map.loc[pp_map["pp_rank_desc"] <= 3].copy()

# target-aware context features for each mapped PP
pp_map["days_since_pp"] = (pp_map["target_race_date"] - pp_map["pp_race_date"]).dt.days
pp_map["distance_gap_furlongs"] = pp_map["target_distance_furlongs"] - pd.to_numeric(pp_map["distance_furlongs"], errors="coerce")
pp_map["abs_distance_gap_furlongs"] = pp_map["distance_gap_furlongs"].abs()
pp_map["same_surface_as_target"] = (pp_map["surface"] == pp_map["target_surface"]).astype("Int64")
pp_map["won_pp"] = (pd.to_numeric(pp_map["official_position"], errors="coerce") == 1).astype("Int64")
pp_map["top3_pp"] = (pd.to_numeric(pp_map["official_position"], errors="coerce") <= 3).astype("Int64")

pp_map_inventory = pd.DataFrame(
    {
        "rows": [len(pp_map)],
        "target_horse_races_with_prior_pp": [pp_map["horse_race_key"].nunique()],
        "target_horse_races_total": [target_base["horse_race_key"].nunique()],
        "max_rank_kept": [pp_map["pp_rank_desc"].max()],
    }
)
display(pp_map_inventory)

,rows,target_horse_races_with_prior_pp,target_horse_races_total,max_rank_kept
0,2897,1013,1054,3


In [14]:
display(pp_map.head(10))

,horse_race_key,registration_number,target_race_date,target_distance_furlongs,target_surface,gps_data_flag,horse_name,pp_track,pp_race_date,pp_race_number,pp_country,race_type,grade,distance_id,distance_unit,about_distance_indicator,distance,surface,post_position,position_at_point_of_call_1,position_at_point_of_call_2,position_at_point_of_call_3,position_at_point_of_call_4,position_at_point_of_call_5,official_position,length_behind_at_poc_1,length_behind_at_poc_2,length_behind_at_poc_3,length_behind_at_poc_4,length_behind_at_poc_5,length_behind_at_finish,length_ahead_at_poc_1,length_ahead_at_poc_2,length_ahead_at_poc_3,length_ahead_at_poc_4,length_ahead_at_poc_5,length_ahead_at_finish,post_time_odds,field_size,purse,is_about_distance,is_graded_race,has_gps_data,distance_furlongs,distance_meters_expected,distance_label,is_nonstandard_distance,pp_race_key,n_active_point_of_calls,early_call_position,second_call_position,mid_call_position,late_call_position,early_call_length_behind,second_call_length_behind,mid_call_length_behind,late_call_length_behind,early_call_length_ahead,late_call_length_ahead,finish_length_behind,finish_length_ahead,early_to_late_call_gain,early_to_finish_gain,late_call_to_finish_gain,length_behind_recovery,pp_rank_desc,days_since_pp,distance_gap_furlongs,abs_distance_gap_furlongs,same_surface_as_target,won_pp,top3_pp
0,LRL|2025-12-12|1|19002714,19002714,2025-12-12,7.0,D,X,Mischief Motion,LRL,2025-11-01,1.0,USA,SOC,NaN,8.0,F,NaN,1M,T,8.0,1.0,1.0,2.0,NaN,4.0,8.0,0.0,0.0,10.0,0.0,310.0,1020.0,250.0,200.0,200.0,0.0,150.0,425.0,677.0,9.0,31950.0,0.0,0.0,1.0,8.0,1609.344,1M,0.0,19002714|LRL|2025-11-01|1,4.0,1.0,1.0,2.0,4.0,0.0,0.0,0.1,3.1,2.5,1.5,10.20,4.25,-3.0,-7.0,-4.0,-10.20,1,41,-1.0,1.0,0,0,0
1,LRL|2025-12-12|1|19002714,19002714,2025-12-12,7.0,D,X,Mischief Motion,LRL,2025-09-26,1.0,USA,SOC,NaN,5.5,F,NaN,5 1/2F,T,6.0,8.0,8.0,NaN,NaN,6.0,7.0,870.0,760.0,0.0,0.0,960.0,950.0,200.0,200.0,0.0,0.0,50.0,275.0,347.0,9.0,31950.0,0.0,0.0,1.0,5.5,1106.424,5 1/2F,0.0,19002714|LRL|2025-09-26|1,3.0,8.0,8.0,8.0,6.0,8.7,7.6,0.0,9.6,2.0,0.5,9.50,2.75,2.0,1.0,-1.0,-0.80,2,77,1.5,1.5,0,0,0
2,LRL|2025-12-12|1|19002714,19002714,2025-12-12,7.0,D,X,Mischief Motion,LRL,2025-09-06,2.0,USA,ALW,NaN,8.5,F,NaN,1 1/16M,D,3.0,3.0,5.0,5.0,NaN,5.0,5.0,250.0,360.0,1210.0,0.0,2400.0,3925.0,10.0,0.0,0.0,0.0,0.0,0.0,217.0,5.0,49000.0,0.0,0.0,1.0,8.5,1709.928,1 1/16M,0.0,19002714|LRL|2025-09-06|2,4.0,3.0,5.0,5.0,5.0,2.5,3.6,12.1,24.0,0.1,0.0,39.25,0.00,-2.0,-2.0,0.0,-36.75,3,97,-1.5,1.5,1,0,0
3,LRL|2025-12-12|1|21003278,21003278,2025-12-12,7.0,D,X,Weekend Wife,LRL,2025-11-14,9.0,USA,ALW,NaN,6.0,F,NaN,6F,D,4.0,6.0,6.0,NaN,NaN,6.0,7.0,610.0,510.0,0.0,0.0,910.0,1025.0,600.0,700.0,0.0,0.0,300.0,0.0,297.0,7.0,48500.0,0.0,0.0,1.0,6.0,1207.008,6F,0.0,21003278|LRL|2025-11-14|9,3.0,6.0,6.0,6.0,6.0,6.1,5.1,0.0,9.1,6.0,3.0,10.25,0.00,0.0,-1.0,-1.0,-4.15,1,28,1.0,1.0,1,0,0
4,LRL|2025-12-12|1|21003278,21003278,2025-12-12,7.0,D,X,Weekend Wife,LRL,2025-09-19,7.0,USA,ALW,NaN,6.0,F,NaN,6F,D,1.0,7.0,8.0,NaN,NaN,8.0,7.0,550.0,1010.0,0.0,0.0,1450.0,1575.0,10.0,0.0,0.0,0.0,0.0,75.0,96.0,8.0,49000.0,0.0,0.0,1.0,6.0,1207.008,6F,0.0,21003278|LRL|2025-09-19|7,3.0,7.0,8.0,8.0,8.0,5.5,10.1,0.0,14.5,0.1,0.0,15.75,0.75,-1.0,0.0,1.0,-10.25,2,84,1.0,1.0,1,0,0
5,LRL|2025-12-12|1|21003278,21003278,2025-12-12,7.0,D,X,Weekend Wife,LRL,2025-04-27,7.0,USA,ALW,NaN,8.0,F,NaN,1M,D,3.0,3.0,3.0,3.0,NaN,4.0,5.0,150.0,200.0,350.0,0.0,650.0,935.0,50.0,50.0,50.0,0.0,10.0,75.0,34.0,7.0,48300.0,0.0,0.0,1.0,8.0,1609.344,1M,0.0,21003278|LRL|2025-04-27|7,4.0,3.0,3.0,3.0,4.0,1.5,2.0,3.5,6.5,0.5,0.1,9.35,0.75,-1.0,-2.0,-1.0,-7.85,3,229,-1.0,1.0,1,0,0
6,LRL|2025-12-12|1|21017739,21017739,2025-12-12,7.0,D,X,Sippin' Time,LRL,2025-11-07,2.0,USA,SOC,NaN,6.0,F,NaN,6F,D,6.0,3.0,6.0,NaN,NaN,6.0,5.0,360.0,700.0,0.0,0.0,1250.0,1475.0,100.0,0.0,0.0,0.0,0.0,250.0,1325.0,7.0,30370.0,0.0,0.0,1.0,6.0,1207.008,6F,0.0,21017739|LRL|2025-11-07|2,3.0,3.0,6.0,6.0,6.0,3.6,7.0,0.0,12.5,1.0,0.0,14.75,2.50,-3.0,-2.0,1.0,

### 4a. Build last-3 traditional sequence features

These preserve the recency order explicitly:

- `last_*`
- `back2_*`
- `back3_*`

This is useful because the most recent start often matters differently from the third-most-recent start.

### Why build sequence features instead of only summaries?

Sequence features keep the order of recent races explicit. That matters because the **most recent race is usually not interchangeable with the third-most-recent race**.

For example, two horses could have the same average finish position over their last three starts, but very different trajectories:

- one may be improving race by race,
- another may be fading,
- and a third may be volatile with one good effort mixed into weaker ones.

A pure mean would blur those patterns together. By preserving `last_*`, `back2_*`, and `back3_*`, the model can learn recency-sensitive patterns such as:

- whether the latest start was especially strong or weak,
- whether a horse is moving closer to today's distance/surface profile,
- whether tactical style has recently changed,
- and whether odds, field size, or class conditions in the latest start differ from the older ones.

I kept only three sequence slots because that is enough to preserve short-run form while still keeping the table manageable.


In [23]:
pp_sequence_cols = [
    "pp_race_date",
    "pp_track",
    "pp_race_number",
    "surface",
    "race_type",
    "distance_label",
    "distance_furlongs",
    "days_since_pp",
    "distance_gap_furlongs",
    "abs_distance_gap_furlongs",
    "same_surface_as_target",
    "official_position",
    "field_size",
    "post_position",
    "post_time_odds",
    "purse",
    "is_about_distance",
    "is_graded_race",
    "has_gps_data",
    "n_active_point_of_calls",
    "early_call_position",
    "mid_call_position",
    "late_call_position",
    "early_call_length_behind",
    "late_call_length_behind",
    "finish_length_behind",
    "early_to_late_call_gain",
    "early_to_finish_gain",
    "length_behind_recovery",
    "pp_race_key",
]

pp_wide = pp_map[["horse_race_key", "pp_rank_desc"] + pp_sequence_cols].copy()
pp_wide["rank_label"] = pp_wide["pp_rank_desc"].map({1: "last", 2: "back2", 3: "back3"})

pp_sequence_parts = []
for col in pp_sequence_cols:
    part = pp_wide.pivot(index="horse_race_key", columns="rank_label", values=col)
    part.columns = [f"{rank}_{col}" for rank in part.columns]
    pp_sequence_parts.append(part)

pp_sequence_features = pd.concat(pp_sequence_parts, axis=1).reset_index()
display(pp_sequence_features.head(10))
print("Traditional sequence feature shape:", pp_sequence_features.shape)

,horse_race_key,back2_pp_race_date,back3_pp_race_date,last_pp_race_date,back2_pp_track,back3_pp_track,last_pp_track,back2_pp_race_number,back3_pp_race_number,last_pp_race_number,back2_surface,back3_surface,last_surface,back2_race_type,back3_race_type,last_race_type,back2_distance_label,back3_distance_label,last_distance_label,back2_distance_furlongs,back3_distance_furlongs,last_distance_furlongs,back2_days_since_pp,back3_days_since_pp,last_days_since_pp,back2_distance_gap_furlongs,back3_distance_gap_furlongs,last_distance_gap_furlongs,back2_abs_distance_gap_furlongs,back3_abs_distance_gap_furlongs,last_abs_distance_gap_furlongs,back2_same_surface_as_target,back3_same_surface_as_target,last_same_surface_as_target,back2_official_position,back3_official_position,last_official_position,back2_field_size,back3_field_size,last_field_size,back2_post_position,back3_post_position,last_post_position,back2_post_time_odds,back3_post_time_odds,last_post_time_odds,back2_purse,back3_purse,last_purse,back2_is_about_distance,back3_is_about_distance,last_is_about_distance,back2_is_graded_race,back3_is_graded_race,last_is_graded_race,back2_has_gps_data,back3_has_gps_data,last_has_gps_data,back2_n_active_point_of_calls,back3_n_active_point_of_calls,last_n_active_point_of_calls,back2_early_call_position,back3_early_call_position,last_early_call_position,back2_mid_call_position,back3_mid_call_position,last_mid_call_position,back2_late_call_position,back3_late_call_position,last_late_call_position,back2_early_call_length_behind,back3_early_call_length_behind,last_early_call_length_behind,back2_late_call_length_behind,back3_late_call_length_behind,last_late_call_length_behind,back2_finish_length_behind,back3_finish_length_behind,last_finish_length_behind,back2_early_to_late_call_gain,back3_early_to_late_call_gain,last_early_to_late_call_gain,back2_early_to_finish_gain,back3_early_to_finish_gain,last_early_to_finish_gain,back2_length_behind_recovery,back3_length_behind_recovery,last_length_behind_recovery,back2_pp_race_key,back3_pp_race_key,last_pp_race_key
0,LRL|2025-12-12|1|19002714,2025-09-26,2025-09-06,2025-11-01,LRL,LRL,LRL,1.0,2.0,1.0,T,D,T,SOC,ALW,SOC,5 1/2F,1 1/16M,1M,5.5,8.50,8.0,77.0,97.0,41.0,1.5,-1.50,-1.0,1.5,1.50,1.0,0,1,0,7.0,5.0,8.0,9.0,5.0,9.0,6.0,3.0,8.0,347.0,217.0,677.0,31950.0,49000.0,31950.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,3.0,4.0,4.0,8.0,3.0,1.0,8.0,5.0,2.0,6.0,5.0,4.0,8.7,2.5,0.0,9.6,24.0,3.1,9.50,39.25,10.20,2.0,-2.0,-3.0,1.0,-2.0,-7.0,-0.80,-36.75,-10.20,19002714|LRL|2025-09-26|1,19002714|LRL|2025-09-06|2,19002714|LRL|2025-11-01|1
1,LRL|2025-12-12|1|21003278,2025-09-19,2025-04-27,2025-11-14,LRL,LRL,LRL,7.0,7.0,9.0,D,D,D,ALW,ALW,ALW,6F,1M,6F,6.0,8.00,6.0,84.0,229.0,28.0,1.0,-1.00,1.0,1.0,1.00,1.0,1,1,1,7.0,5.0,7.0,8.0,7.0,7.0,1.0,3.0,4.0,96.0,34.0,297.0,49000.0,48300.0,48500.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,3.0,4.0,3.0,7.0,3.0,6.0,8.0,3.0,6.0,8.0,4.0,6.0,5.5,1.5,6.1,14.5,6.5,9.1,15.75,9.35,10.25,-1.0,-1.0,0.0,0.0,-2.0,-1.0,-10.25,-7.85,-4.15,21003278|LRL|2025-09-19|7,21003278|LRL|2025-04-27|7,21003278|LRL|2025-11-14|9
2,LRL|2025-12-12|1|21017739,2025-10-25,2025-07-05,2025-11-07,LRL,DEL,LRL,7.0,4.0,2.0,D,D,D,CLM,SOC,SOC,6F,6F,6F,6.0,6.00,6.0,48.0,160.0,35.0,1.0,1.00,1.0,1.0,1.00,1.0,1,1,1,6.0,7.0,5.0,8.0,7.0,7.0,5.0,6.0,6.0,201.0,832.0,1325.0,25700.0,25000.0,30370.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,3.0,3.0,3.0,7.0,5.0,3.0,8.0,7.0,6.0,8.0,7.0,6.0,6.7,3.0,3.6,10.6,7.0,12.5,15.00,7.35,14.75,-1.0,-2.0,-3.0,1.0,-2.0,-2.0,-8.30,-4.35,-11.15,21017739|LRL|2025-10-25|7,21017739|DEL|2025-07-05|4,21017739|LRL|2025-11-07|2
3,LRL|2025-12-12|1|22002156,2025-11-08,2025-10-10,2025-11-22,LRL,LRL,LRL,6.0,3.0,1.0,T,T,T,MCL,MCL,SOC,5 1/2F,1M,5 1/2F,5.5,8.00,5.5,34.0,63.0,20.0,1.5,-1.00,1.5,1.5,1.00,1.5,0,0,0,1.0,7.0,7.0,12.0,11.0,9.0,12.0,3.0,9.0,24.0,18.0,40.0,28100.0,32500.0,28040.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,3.0,4.0,3.0,5.0,5.0,6.0,4.0,7.0,9.0,3.0,8.0,7.0,4.1,5.0,3.6,5.5,6.1,9.1,0.00,5.55,6.75,2.0,-3.0,-1.0,4.0,-2.0,-1.0,4.1

Traditional sequence feature shape: (1013, 91)


### 4b. Build summary traditional features

These summarize a horse's recent traditional history into stable numerical signals.

### Why build summary traditional features as well?

Summary features complement the sequence features. The sequence columns preserve detail, but they can be noisy, sparse, and high-dimensional. Summary features compress recent history into more stable signals that many models can use effectively.

The summaries here were chosen to represent a few different ideas:

- **Experience / recency:** number of starts and days since last start capture how much recent evidence we have and whether the horse is coming back quickly or off a layoff.
- **Outcome quality:** finish-position summaries, win rate, and top-3 rate give a compact measure of recent competitiveness.
- **Market information:** average odds provide a rough crowd-based signal of perceived strength.
- **Race context:** field size, purse, same-surface share, and distance-gap summaries help describe whether the horse has been racing in conditions similar to today's race.
- **Running style:** early/late positions, gains, and margin-recovery features summarize whether the horse tends to show speed early, close late, or lose ground through the race.

In other words, the summary table is meant to give the model a cleaner estimate of *what kind of recent horse this is*, while the sequence table preserves the exact last few race snapshots.


In [22]:
traditional_summary_features = (
    pp_map.groupby("horse_race_key")
    .agg(
        pp_num_prior_starts=("pp_race_key", "nunique"),
        pp_days_since_last_start=("days_since_pp", "min"),
        pp_finish_mean=("official_position", "mean"),
        pp_finish_best=("official_position", "min"),
        pp_finish_worst=("official_position", "max"),
        pp_finish_std=("official_position", "std"),
        pp_win_rate=("won_pp", "mean"),
        pp_top3_rate=("top3_pp", "mean"),
        pp_post_time_odds_mean=("post_time_odds", "mean"),
        pp_post_time_odds_min=("post_time_odds", "min"),
        pp_field_size_mean=("field_size", "mean"),
        pp_purse_mean=("purse", "mean"),
        pp_purse_max=("purse", "max"),
        pp_same_surface_share=("same_surface_as_target", "mean"),
        pp_abs_distance_gap_mean=("abs_distance_gap_furlongs", "mean"),
        pp_abs_distance_gap_min=("abs_distance_gap_furlongs", "min"),
        pp_has_gps_history_share=("has_gps_data", "mean"),
        pp_num_active_pocs_mean=("n_active_point_of_calls", "mean"),
        pp_early_call_position_mean=("early_call_position", "mean"),
        pp_late_call_position_mean=("late_call_position", "mean"),
        pp_early_to_late_gain_mean=("early_to_late_call_gain", "mean"),
        pp_early_to_finish_gain_mean=("early_to_finish_gain", "mean"),
        pp_early_length_behind_mean=("early_call_length_behind", "mean"),
        pp_late_length_behind_mean=("late_call_length_behind", "mean"),
        pp_finish_length_behind_mean=("finish_length_behind", "mean"),
        pp_length_recovery_mean=("length_behind_recovery", "mean"),
    )
    .reset_index()
)

traditional_feature_table = traditional_summary_features.merge(
    pp_sequence_features,
    on="horse_race_key",
    how="left",
)

display(traditional_feature_table.head(10))
print("Traditional feature table shape:", traditional_feature_table.shape)

,horse_race_key,pp_num_prior_starts,pp_days_since_last_start,pp_finish_mean,pp_finish_best,pp_finish_worst,pp_finish_std,pp_win_rate,pp_top3_rate,pp_post_time_odds_mean,pp_post_time_odds_min,pp_field_size_mean,pp_purse_mean,pp_purse_max,pp_same_surface_share,pp_abs_distance_gap_mean,pp_abs_distance_gap_min,pp_has_gps_history_share,pp_num_active_pocs_mean,pp_early_call_position_mean,pp_late_call_position_mean,pp_early_to_late_gain_mean,pp_early_to_finish_gain_mean,pp_early_length_behind_mean,pp_late_length_behind_mean,pp_finish_length_behind_mean,pp_length_recovery_mean,back2_pp_race_date,back3_pp_race_date,last_pp_race_date,back2_pp_track,back3_pp_track,last_pp_track,back2_pp_race_number,back3_pp_race_number,last_pp_race_number,back2_surface,back3_surface,last_surface,back2_race_type,back3_race_type,last_race_type,back2_distance_label,back3_distance_label,last_distance_label,back2_distance_furlongs,back3_distance_furlongs,last_distance_furlongs,back2_days_since_pp,back3_days_since_pp,last_days_since_pp,back2_distance_gap_furlongs,back3_distance_gap_furlongs,last_distance_gap_furlongs,back2_abs_distance_gap_furlongs,back3_abs_distance_gap_furlongs,last_abs_distance_gap_furlongs,back2_same_surface_as_target,back3_same_surface_as_target,last_same_surface_as_target,back2_official_position,back3_official_position,last_official_position,back2_field_size,back3_field_size,last_field_size,back2_post_position,back3_post_position,last_post_position,back2_post_time_odds,back3_post_time_odds,last_post_time_odds,back2_purse,back3_purse,last_purse,back2_is_about_distance,back3_is_about_distance,last_is_about_distance,back2_is_graded_race,back3_is_graded_race,last_is_graded_race,back2_has_gps_data,back3_has_gps_data,last_has_gps_data,back2_n_active_point_of_calls,back3_n_active_point_of_calls,last_n_active_point_of_calls,back2_early_call_position,back3_early_call_position,last_early_call_position,back2_mid_call_position,back3_mid_call_position,last_mid_call_position,back2_late_call_position,back3_late_call_position,last_late_call_position,back2_early_call_length_behind,back3_early_call_length_behind,last_early_call_length_behind,back2_late_call_length_behind,back3_late_call_length_behind,last_late_call_length_behind,back2_finish_length_behind,back3_finish_length_behind,last_finish_length_behind,back2_early_to_late_call_gain,back3_early_to_late_call_gain,last_early_to_late_call_gain,back2_early_to_finish_gain,back3_early_to_finish_gain,last_early_to_finish_gain,back2_length_behind_recovery,back3_length_behind_recovery,last_length_behind_recovery,back2_pp_race_key,back3_pp_race_key,last_pp_race_key
0,LRL|2025-12-12|1|19002714,3,41,6.666667,5.0,8.0,1.527525,0.0,0.0,413.666667,217.0,7.666667,37633.333333,49000.0,0.333333,1.333333,1.0,1.000000,3.666667,4.000000,5.000000,-1.000000,-2.666667,3.733333,12.233333,19.650000,-15.916667,2025-09-26,2025-09-06,2025-11-01,LRL,LRL,LRL,1.0,2.0,1.0,T,D,T,SOC,ALW,SOC,5 1/2F,1 1/16M,1M,5.5,8.50,8.0,77.0,97.0,41.0,1.5,-1.50,-1.0,1.5,1.50,1.0,0,1,0,7.0,5.0,8.0,9.0,5.0,9.0,6.0,3.0,8.0,347.0,217.0,677.0,31950.0,49000.0,31950.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,3.0,4.0,4.0,8.0,3.0,1.0,8.0,5.0,2.0,6.0,5.0,4.0,8.7,2.5,0.0,9.6,24.0,3.1,9.50,39.25,10.20,2.0,-2.0,-3.0,1.0,-2.0,-7.0,-0.80,-36.75,-10.20,19002714|LRL|2025-09-26|1,19002714|LRL|2025-09-06|2,19002714|LRL|2025-11-01|1
1,LRL|2025-12-12|1|21003278,3,28,6.333333,5.0,7.0,1.154701,0.0,0.0,142.333333,34.0,7.333333,48600.000000,49000.0,1.0,1.000000,1.0,1.000000,3.333333,5.333333,6.000000,-0.666667,-1.000000,4.366667,10.033333,11.783333,-7.416667,2025-09-19,2025-04-27,2025-11-14,LRL,LRL,LRL,7.0,7.0,9.0,D,D,D,ALW,ALW,ALW,6F,1M,6F,6.0,8.00,6.0,84.0,229.0,28.0,1.0,-1.00,1.0,1.0,1.00,1.0,1,1,1,7.0,5.0,7.0,8.0,7.0,7.0,1.0,3.0,4.0,96.0,34.0,297.0,49000.0,48300.0,48500.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,3.0,4.0,3.0,7.0,3.0,6.0,8.0,3.0,6.0,8.0,4.0,6.0,5.5,1.5,6.1,14.5,6.5,9.1,15.75,9.35,10.25,-1.0,-1.0,0.0,0.0,-2.0,-1.0,-10.25,-7.85,-4.15,21003278|LRL|2025-09-19|7,21003278|

Traditional feature table shape: (1013, 117)


## 5. Historical GPS feature engineering

The GPS PP table is still gate-level, so we first summarize each historical GPS PP race into **one row per horse prior race**.

Then we map those summarized historical GPS races back to each Laurel target horse-race and again keep the **three most recent**.

### Why summarize GPS data to one row per prior race first?

The raw GPS table is gate-level, which is richer than traditional PPs but also too granular to merge directly into a horse-race level modeling table. If we joined the raw gate rows as-is, one historical race would contribute many rows and distort the unit of analysis.

So the first step is to convert each historical GPS race into a single race-level summary. That lets us keep the GPS signal while preserving a clean structure:

- one row per historical horse-race,
- then one set of historical races per target Laurel horse-race,
- then one final row per target horse-race in the modeling table.

This is also conceptually useful: the goal is not to predict from each individual gate snapshot, but to use the gate snapshots to describe broader properties of a prior performance, such as pace profile, position changes, efficiency, and ground loss.


In [25]:
gps_pp_race_summary = (
    gps_pp_clean
    .sort_values(["registration_number", "pp_race_date", "pp_race_number", "gate"], ascending=[True, False, False, False])
    .groupby("pp_race_key", group_keys=False)
    .apply(summarize_single_gps_race)
    .reset_index(drop=True)
);

display(gps_pp_race_summary.head(10))
print("Historical GPS prior-race summary shape:", gps_pp_race_summary.shape)

/var/folders/10/yh5g9ksn5fb_2pzdwtzd2tlh0000gn/T/ipykernel_89883/1150147293.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize_single_gps_race)


,horse_name,registration_number,pp_track,pp_race_date,pp_race_number,pp_race_key,surface,race_type,distance_label,distance_furlongs,distance_meters_expected,purse,field_size,post_position,official_position,num_gate_rows,early_position,late_position,best_position,worst_position,mean_position,std_position,position_gain_early_to_late,sectional_time_mean,sectional_time_std,sectional_time_early_mean,sectional_time_late_mean,late_minus_early_sectional,distance_behind_early,distance_behind_late,distance_behind_mean,distance_behind_max,distance_behind_recovery,time_behind_mean,final_running_time,cumulative_distance_final,cumulative_strides_final,distance_efficiency_ratio,extra_distance_m,meters_per_stride
0,Bode's Maker,15000089,LRL,2025-10-04,9,15000089|LRL|2025-10-04|9,T,SOC,1M,8.0,1609.344,25520,9,9,1,16,7.000000,4.333333,1,8,6.062500,2.235136,2.666667,5.977500,0.396459,6.208333,5.810000,-0.398333,20.233333,5.350000,15.281250,24.7,14.883333,0.896688,95.600,1618.1,226.2,1.005441,8.756,7.153404
1,Bode's Maker,15000089,LRL,2025-11-01,6,15000089|LRL|2025-11-01|6,T,SOC,1 1/16M,8.5,1709.928,27020,12,6,2,17,11.500000,5.500000,2,12,8.647059,3.219883,6.000000,6.100588,0.409275,6.155000,6.163333,0.008333,16.833333,5.283333,10.658824,23.9,11.550000,0.632000,103.708,1724.6,244.3,1.008580,14.672,7.059353
2,Bode's Maker,15000089,LRL,2025-11-22,2,15000089|LRL|2025-11-22|2,T,SOC,1M,8.0,1609.344,26520,11,7,4,16,9.166667,6.500000,4,10,7.687500,1.662077,2.666667,6.151250,0.372843,6.158333,6.273333,0.115000,19.450000,7.583333,13.518750,21.6,11.866667,0.813125,98.442,1624.8,229.1,1.009604,15.456,7.092100
3,Tiberius Mercurius,16009053,AQU,2025-12-10,6,16009053|AQU|2025-12-10|6,D,CLM,1M,8.0,1609.344,28000,7,1,4,16,2.000000,2.500000,2,4,2.187500,0.543906,-0.500000,6.353750,0.586161,5.971667,6.905000,0.933333,1.266667,7.250000,3.787500,16.1,-5.983333,0.256437,101.637,1613.8,228.1,1.002769,4.456,7.074967
4,Tiberius Mercurius,16009053,AQU,2025-12-31,8,16009053|AQU|2025-12-31|8,D,CLM,6 1/2F,6.5,1307.592,28000,11,4,10,13,11.000000,10.800000,10,11,10.923077,0.277350,0.200000,6.233077,0.435036,5.996000,6.562000,0.566000,17.600000,21.980000,21.100000,26.1,-4.380000,1.296231,81.001,1312.7,181.6,1.003906,5.108,7.228524
5,Tiberius Mercurius,16009053,BAQ,2025-09-18,6,16009053|BAQ|2025-09-18|6,D,CLM,1 1/8M,9.0,1810.512,32000,8,7,7,18,8.000000,7.000000,7,8,7.666667,0.485071,1.000000,6.351111,0.510258,6.460000,6.695000,0.235000,33.166667,29.816667,33.688889,43.5,3.350000,2.081389,114.321,1824.4,254.1,1.007671,13.888,7.179850
6,Armando R,16011638,LRL,2025-09-21,4,16011638|LRL|2025-09-21|4,D,SOC,1 1/16M,8.5,1709.928,30370,7,1,1,17,5.833333,3.500000,1,7,5.117647,1.932691,2.333333,6.234706,0.255444,6.256667,6.290000,0.033333,18.366667,5.266667,12.952941,20.8,13.100000,0.792235,106.010,1712.0,226.5,1.001212,2.072,7.558499
7,Armando R,16011638,LRL,2025-10-18,2,16011638|LRL|2025-10-18|2,D,SOC,1 1/8M,9.0,1810.512,29435,6,4,1,18,6.000000,2.500000,1,6,4.833333,1.855041,3.500000,6.220556,0.329964,6.288333,6.318333,0.030000,13.833333,5.233333,11.144444,16.7,8.600000,0.677056,111.970,1814.9,241.4,1.002424,4.388,7.518227
8,Armando R,16011638,LRL,2025-11-29,5,16011638|LRL|2025-11-29|5,D,SOC,1 1/16M,8.5,1709.928,31370,9,3,1,17,6.833333,4.333333,1,9,6.470588,2.426872,2.500000,6.280588,0.298611,6.235000,6.375000,0.140000,11.400000,9.316667,13.364706,21.3,2.083333,0.832118,106.770,1716.7,227.6,1.003960,6.772,7.542619
9,Bold Endeavor,16012148,BAQ,2025-09-18,6,16012148|BAQ|2025-09-18|6,D,CLM,1 1/8M,9.0,1810.512,32000,8,2,4,18,3.166667,4.000000,3,4,3.611111,0.501631,-0.833333,6.229444,0.314277,6.090000,6.566667,0.476667,5.350000,9.050000,7.283333,11.3,-3.700000,0.454611,112.131,1815.7,249.6,1.002865,5.188,7.274439


Historical GPS prior-race summary shape: (1739, 40)


In [29]:
gps_map = pp_map[
    ["horse_race_key", "pp_rank_desc", "pp_race_key", "target_race_date", "target_distance_furlongs", "target_surface"]
].merge(
    gps_pp_race_summary,
    on="pp_race_key",
    how="left",
)

gps_map = gps_map.loc[gps_map["pp_race_date"].notna()].copy()

gps_map["days_since_gps_pp"] = (gps_map["target_race_date"] - gps_map["pp_race_date"]).dt.days
gps_map["distance_gap_furlongs"] = gps_map["target_distance_furlongs"] - pd.to_numeric(gps_map["distance_furlongs"], errors="coerce")
gps_map["abs_distance_gap_furlongs"] = gps_map["distance_gap_furlongs"].abs()
gps_map["same_surface_as_target"] = (gps_map["surface"] == gps_map["target_surface"]).astype("Int64")
gps_map["won_gps_pp"] = (pd.to_numeric(gps_map["official_position"], errors="coerce") == 1).astype("Int64")
gps_map["top3_gps_pp"] = (pd.to_numeric(gps_map["official_position"], errors="coerce") <= 3).astype("Int64")

gps_map_inventory = pd.DataFrame(
    {
        "rows": [len(gps_map)],
        "target_horse_races_with_historical_gps": [gps_map["horse_race_key"].nunique()],
        "target_horse_races_total": [target_base["horse_race_key"].nunique()],
        "unique_historical_gps_prior_races": [gps_map["pp_race_key"].nunique()],
    }
)
display(gps_map_inventory)

,rows,target_horse_races_with_historical_gps,target_horse_races_total,unique_historical_gps_prior_races
0,2270,926,1054,1739


In [30]:
display(gps_map.head(10))

,horse_race_key,pp_rank_desc,pp_race_key,target_race_date,target_distance_furlongs,target_surface,horse_name,registration_number,pp_track,pp_race_date,pp_race_number,surface,race_type,distance_label,distance_furlongs,distance_meters_expected,purse,field_size,post_position,official_position,num_gate_rows,early_position,late_position,best_position,worst_position,mean_position,std_position,position_gain_early_to_late,sectional_time_mean,sectional_time_std,sectional_time_early_mean,sectional_time_late_mean,late_minus_early_sectional,distance_behind_early,distance_behind_late,distance_behind_mean,distance_behind_max,distance_behind_recovery,time_behind_mean,final_running_time,cumulative_distance_final,cumulative_strides_final,distance_efficiency_ratio,extra_distance_m,meters_per_stride,days_since_gps_pp,distance_gap_furlongs,abs_distance_gap_furlongs,same_surface_as_target,won_gps_pp,top3_gps_pp
0,LRL|2025-12-12|1|19002714,1,19002714|LRL|2025-11-01|1,2025-12-12,7.0,D,Mischief Motion,19002714,LRL,2025-11-01,1.0,T,SOC,1M,8.0,1609.344,31950.0,9.0,8.0,8.0,16.0,1.666667,3.50,1.0,8.0,2.375000,1.784190,-1.833333,6.238750,0.323045,6.111667,6.370000,0.258333,1.050000,10.333333,4.412500,28.2,-9.283333,0.272000,99.833,1612.3,237.3,1.001837,2.956,6.794353,41,-1.0,1.0,0,0,0
1,LRL|2025-12-12|1|19002714,2,19002714|LRL|2025-09-26|1,2025-12-12,7.0,D,Mischief Motion,19002714,LRL,2025-09-26,1.0,T,SOC,5 1/2F,5.5,1106.424,31950.0,9.0,6.0,7.0,11.0,6.750000,6.75,5.0,8.0,7.090909,0.943880,0.000000,5.850909,0.490111,5.815000,5.970000,0.155000,10.175000,24.650000,17.172727,26.7,-14.475000,0.994182,64.376,1107.1,159.1,1.000611,0.676,6.958517,77,1.5,1.5,0,0,0
2,LRL|2025-12-12|1|19002714,3,19002714|LRL|2025-09-06|2,2025-12-12,7.0,D,Mischief Motion,19002714,LRL,2025-09-06,2.0,D,ALW,1 1/16M,8.5,1709.928,49000.0,5.0,3.0,5.0,17.0,3.333333,5.00,3.0,5.0,4.294118,0.848875,-1.666667,6.651765,0.638683,6.365000,7.325000,0.960000,2.866667,52.450000,22.194118,89.4,-49.583333,1.512471,113.062,1715.4,262.8,1.003200,5.472,6.527397,97,-1.5,1.5,1,0,0
3,LRL|2025-12-12|1|21003278,1,21003278|LRL|2025-11-14|9,2025-12-12,7.0,D,Weekend Wife,21003278,LRL,2025-11-14,9.0,D,ALW,6F,6.0,1207.008,48500.0,7.0,4.0,7.0,12.0,6.000000,6.25,6.0,7.0,6.083333,0.288675,-0.250000,6.137500,0.494039,5.980000,6.490000,0.510000,9.425000,21.850000,15.158333,26.8,-12.425000,0.925333,73.629,1209.6,174.5,1.002147,2.592,6.931805,28,1.0,1.0,1,0,0
4,LRL|2025-12-12|1|21003278,2,21003278|LRL|2025-09-19|7,2025-12-12,7.0,D,Weekend Wife,21003278,LRL,2025-09-19,7.0,D,ALW,6F,6.0,1207.008,49000.0,8.0,1.0,7.0,12.0,4.750000,7.75,4.0,8.0,6.333333,1.497473,-3.000000,6.152500,0.411960,5.927500,6.475000,0.547500,5.575000,34.450000,17.858333,41.1,-28.875000,1.108833,73.819,1208.5,177.4,1.001236,1.492,6.812289,84,1.0,1.0,1,0,0
5,LRL|2025-12-12|1|21003278,3,21003278|LRL|2025-04-27|7,2025-12-12,7.0,D,Weekend Wife,21003278,LRL,2025-04-27,7.0,D,ALW,1M,8.0,1609.344,48300.0,7.0,3.0,5.0,16.0,1.000000,4.00,1.0,5.0,2.250000,1.612452,-3.000000,6.249375,0.460572,5.940000,6.608333,0.668333,0.000000,12.666667,4.925000,21.9,-12.666667,0.334313,99.989,1613.0,240.9,1.002272,3.656,6.695724,229,-1.0,1.0,1,0,0
6,LRL|2025-12-12|1|21017739,1,21017739|LRL|2025-11-07|2,2025-12-12,7.0,D,Sippin' Time,21017739,LRL,2025-11-07,2.0,D,SOC,6F,6.0,1207.008,30370.0,7.0,6.0,5.0,12.0,1.500000,5.75,1.0,6.0,3.416667,2.020726,-4.250000,6.110833,0.619420,5.555000,6.770000,1.215000,0.125000,29.850000,11.825000,36.3,-29.725000,0.772417,73.310,1218.2,174.2,1.009273,11.192,6.993111,35,1.0,1.0,1,0,0
7,LRL|2025-12-12|1|21017739,2,21017739|LRL|2025-10-25|7,2025-12-12,7.0,D,Sippin' Time,21017739,LRL,2025-10-25,7.0,D,CLM,6F,6.0,1207.008,25700.0,8.0,5.0,6.0,12.0,8.000000,7.50,6.0,8.0,7.833333,0.577350,0.500000,6.265833,0.602728,6.045000,6.652500,0.607500,14.000000,33.600000,21.183333,37.7,-19.600000,1.327500,75.161,1210.1,172.9,1.002562,3.092,6.998843,48,1.0,1.0,1,0,0
9,LRL|2025-12-12|1|22002156,1,22002156|LRL|2025-11-22|1,2025-12-12,7.0,D,Mun Mun Can Run,22002156,LRL,2025-

### 5a. Build last-3 GPS sequence features

### Why keep last-3 GPS sequence features?

The GPS sequence features play the same role as the traditional sequence features, but they preserve a much richer description of *how* a horse ran, not just *where it finished*.

This matters because two horses can finish in similar positions while producing very different race shapes. GPS-derived sequence features help preserve things like:

- whether the horse moved forward or backward through the race,
- whether it lost or saved ground,
- whether it ran efficiently relative to the listed distance,
- whether it had stronger early versus late sectionals,
- and whether the latest GPS start shows a different movement pattern than the prior two.

Keeping these as `last_*`, `back2_*`, and `back3_*` gives the later model a chance to learn whether the most recent movement profile is especially predictive.


In [32]:
gps_sequence_cols = [
    "pp_race_date",
    "pp_track",
    "pp_race_number",
    "surface",
    "race_type",
    "distance_label",
    "distance_furlongs",
    "days_since_gps_pp",
    "distance_gap_furlongs",
    "abs_distance_gap_furlongs",
    "same_surface_as_target",
    "official_position",
    "field_size",
    "post_position",
    "purse",
    "num_gate_rows",
    "early_position",
    "late_position",
    "best_position",
    "worst_position",
    "mean_position",
    "std_position",
    "position_gain_early_to_late",
    "sectional_time_mean",
    "sectional_time_std",
    "sectional_time_early_mean",
    "sectional_time_late_mean",
    "late_minus_early_sectional",
    "distance_behind_early",
    "distance_behind_late",
    "distance_behind_mean",
    "distance_behind_max",
    "distance_behind_recovery",
    "time_behind_mean",
    "final_running_time",
    "cumulative_distance_final",
    "cumulative_strides_final",
    "distance_efficiency_ratio",
    "extra_distance_m",
    "meters_per_stride",
    "pp_race_key",
]

gps_wide = gps_map[["horse_race_key", "pp_rank_desc"] + gps_sequence_cols].copy()
gps_wide["rank_label"] = gps_wide["pp_rank_desc"].map({1: "last", 2: "back2", 3: "back3"})

gps_sequence_parts = []
for col in gps_sequence_cols:
    part = gps_wide.pivot(index="horse_race_key", columns="rank_label", values=col)
    part.columns = [f"{rank}_{col}" for rank in part.columns]
    gps_sequence_parts.append(part)

gps_sequence_features = pd.concat(gps_sequence_parts, axis=1).reset_index()
display(gps_sequence_features.head(10))
print("GPS sequence feature shape:", gps_sequence_features.shape)

,horse_race_key,back2_pp_race_date,back3_pp_race_date,last_pp_race_date,back2_pp_track,back3_pp_track,last_pp_track,back2_pp_race_number,back3_pp_race_number,last_pp_race_number,back2_surface,back3_surface,last_surface,back2_race_type,back3_race_type,last_race_type,back2_distance_label,back3_distance_label,last_distance_label,back2_distance_furlongs,back3_distance_furlongs,last_distance_furlongs,back2_days_since_gps_pp,back3_days_since_gps_pp,last_days_since_gps_pp,back2_distance_gap_furlongs,back3_distance_gap_furlongs,last_distance_gap_furlongs,back2_abs_distance_gap_furlongs,back3_abs_distance_gap_furlongs,last_abs_distance_gap_furlongs,back2_same_surface_as_target,back3_same_surface_as_target,last_same_surface_as_target,back2_official_position,back3_official_position,last_official_position,back2_field_size,back3_field_size,last_field_size,back2_post_position,back3_post_position,last_post_position,back2_purse,back3_purse,last_purse,back2_num_gate_rows,back3_num_gate_rows,last_num_gate_rows,back2_early_position,back3_early_position,last_early_position,back2_late_position,back3_late_position,last_late_position,back2_best_position,back3_best_position,last_best_position,back2_worst_position,back3_worst_position,last_worst_position,back2_mean_position,back3_mean_position,last_mean_position,back2_std_position,back3_std_position,last_std_position,back2_position_gain_early_to_late,back3_position_gain_early_to_late,last_position_gain_early_to_late,back2_sectional_time_mean,back3_sectional_time_mean,last_sectional_time_mean,back2_sectional_time_std,back3_sectional_time_std,last_sectional_time_std,back2_sectional_time_early_mean,back3_sectional_time_early_mean,last_sectional_time_early_mean,back2_sectional_time_late_mean,back3_sectional_time_late_mean,last_sectional_time_late_mean,back2_late_minus_early_sectional,back3_late_minus_early_sectional,last_late_minus_early_sectional,back2_distance_behind_early,back3_distance_behind_early,last_distance_behind_early,back2_distance_behind_late,back3_distance_behind_late,last_distance_behind_late,back2_distance_behind_mean,back3_distance_behind_mean,last_distance_behind_mean,back2_distance_behind_max,back3_distance_behind_max,last_distance_behind_max,back2_distance_behind_recovery,back3_distance_behind_recovery,last_distance_behind_recovery,back2_time_behind_mean,back3_time_behind_mean,last_time_behind_mean,back2_final_running_time,back3_final_running_time,last_final_running_time,back2_cumulative_distance_final,back3_cumulative_distance_final,last_cumulative_distance_final,back2_cumulative_strides_final,back3_cumulative_strides_final,last_cumulative_strides_final,back2_distance_efficiency_ratio,back3_distance_efficiency_ratio,last_distance_efficiency_ratio,back2_extra_distance_m,back3_extra_distance_m,last_extra_distance_m,back2_meters_per_stride,back3_meters_per_stride,last_meters_per_stride,back2_pp_race_key,back3_pp_race_key,last_pp_race_key
0,LRL|2025-12-12|1|19002714,2025-09-26,2025-09-06,2025-11-01,LRL,LRL,LRL,1.0,2.0,1.0,T,D,T,SOC,ALW,SOC,5 1/2F,1 1/16M,1M,5.5,8.5,8.0,77.0,97.0,41.0,1.5,-1.5,-1.0,1.5,1.5,1.0,0,1,0,7.0,5.0,8.0,9.0,5.0,9.0,6.0,3.0,8.0,31950.0,49000.0,31950.0,11.0,17.0,16.0,6.750000,3.333333,1.666667,6.750000,5.000000,3.500000,5.0,3.0,1.0,8.0,5.0,8.0,7.090909,4.294118,2.375000,0.943880,0.848875,1.784190,0.000000,-1.666667,-1.833333,5.850909,6.651765,6.238750,0.490111,0.638683,0.323045,5.815000,6.365000,6.111667,5.970000,7.325000,6.370000,0.155000,0.960000,0.258333,10.175000,2.866667,1.050000,24.650000,52.450000,10.333333,17.172727,22.194118,4.412500,26.7,89.4,28.2,-14.475000,-49.583333,-9.283333,0.994182,1.512471,0.272000,64.376,113.062,99.833,1107.1,1715.4,1612.3,159.1,262.8,237.3,1.000611,1.003200,1.001837,0.676,5.472,2.956,6.958517,6.527397,6.794353,19002714|LRL|2025-09-26|1,19002714|LRL|2025-09-06|2,19002714|LRL|2025-11-01|1
1,LRL|2025-12-12|1|21003278,2025-09-19,2025-04-27,2025-11-14,LRL,LRL,LRL,7.0,7.0,9.0,D,D,D,ALW,ALW,ALW,6F,1M,6F,6.0,8.0,6.0,84.0,229.0,28.0,1.0,-1.

GPS sequence feature shape: (926, 124)


### 5a. Build last-3 GPS sequence features

### Why keep last-3 GPS sequence features?

The GPS sequence features play the same role as the traditional sequence features, but they preserve a much richer description of *how* a horse ran, not just *where it finished*.

This matters because two horses can finish in similar positions while producing very different race shapes. GPS-derived sequence features help preserve things like:

- whether the horse moved forward or backward through the race,
- whether it lost or saved ground,
- whether it ran efficiently relative to the listed distance,
- whether it had stronger early versus late sectionals,
- and whether the latest GPS start shows a different movement pattern than the prior two.

Keeping these as `last_*`, `back2_*`, and `back3_*` gives the later model a chance to learn whether the most recent movement profile is especially predictive.


In [34]:
gps_sequence_cols = [
    "pp_race_date",
    "pp_track",
    "pp_race_number",
    "surface",
    "race_type",
    "distance_label",
    "distance_furlongs",
    "days_since_gps_pp",
    "distance_gap_furlongs",
    "abs_distance_gap_furlongs",
    "same_surface_as_target",
    "official_position",
    "field_size",
    "post_position",
    "purse",
    "num_gate_rows",
    "early_position",
    "late_position",
    "best_position",
    "worst_position",
    "mean_position",
    "std_position",
    "position_gain_early_to_late",
    "sectional_time_mean",
    "sectional_time_std",
    "sectional_time_early_mean",
    "sectional_time_late_mean",
    "late_minus_early_sectional",
    "distance_behind_early",
    "distance_behind_late",
    "distance_behind_mean",
    "distance_behind_max",
    "distance_behind_recovery",
    "time_behind_mean",
    "final_running_time",
    "cumulative_distance_final",
    "cumulative_strides_final",
    "distance_efficiency_ratio",
    "extra_distance_m",
    "meters_per_stride",
    "pp_race_key",
]

gps_wide = gps_map[["horse_race_key", "pp_rank_desc"] + gps_sequence_cols].copy()
gps_wide["rank_label"] = gps_wide["pp_rank_desc"].map({1: "last", 2: "back2", 3: "back3"})

gps_sequence_parts = []
for col in gps_sequence_cols:
    part = gps_wide.pivot(index="horse_race_key", columns="rank_label", values=col)
    part.columns = [f"{rank}_{col}" for rank in part.columns]
    gps_sequence_parts.append(part)

gps_sequence_features = pd.concat(gps_sequence_parts, axis=1).reset_index()
display(gps_sequence_features.head(10))
print("GPS sequence feature shape:", gps_sequence_features.shape)

,horse_race_key,back2_pp_race_date,back3_pp_race_date,last_pp_race_date,back2_pp_track,back3_pp_track,last_pp_track,back2_pp_race_number,back3_pp_race_number,last_pp_race_number,back2_surface,back3_surface,last_surface,back2_race_type,back3_race_type,last_race_type,back2_distance_label,back3_distance_label,last_distance_label,back2_distance_furlongs,back3_distance_furlongs,last_distance_furlongs,back2_days_since_gps_pp,back3_days_since_gps_pp,last_days_since_gps_pp,back2_distance_gap_furlongs,back3_distance_gap_furlongs,last_distance_gap_furlongs,back2_abs_distance_gap_furlongs,back3_abs_distance_gap_furlongs,last_abs_distance_gap_furlongs,back2_same_surface_as_target,back3_same_surface_as_target,last_same_surface_as_target,back2_official_position,back3_official_position,last_official_position,back2_field_size,back3_field_size,last_field_size,back2_post_position,back3_post_position,last_post_position,back2_purse,back3_purse,last_purse,back2_num_gate_rows,back3_num_gate_rows,last_num_gate_rows,back2_early_position,back3_early_position,last_early_position,back2_late_position,back3_late_position,last_late_position,back2_best_position,back3_best_position,last_best_position,back2_worst_position,back3_worst_position,last_worst_position,back2_mean_position,back3_mean_position,last_mean_position,back2_std_position,back3_std_position,last_std_position,back2_position_gain_early_to_late,back3_position_gain_early_to_late,last_position_gain_early_to_late,back2_sectional_time_mean,back3_sectional_time_mean,last_sectional_time_mean,back2_sectional_time_std,back3_sectional_time_std,last_sectional_time_std,back2_sectional_time_early_mean,back3_sectional_time_early_mean,last_sectional_time_early_mean,back2_sectional_time_late_mean,back3_sectional_time_late_mean,last_sectional_time_late_mean,back2_late_minus_early_sectional,back3_late_minus_early_sectional,last_late_minus_early_sectional,back2_distance_behind_early,back3_distance_behind_early,last_distance_behind_early,back2_distance_behind_late,back3_distance_behind_late,last_distance_behind_late,back2_distance_behind_mean,back3_distance_behind_mean,last_distance_behind_mean,back2_distance_behind_max,back3_distance_behind_max,last_distance_behind_max,back2_distance_behind_recovery,back3_distance_behind_recovery,last_distance_behind_recovery,back2_time_behind_mean,back3_time_behind_mean,last_time_behind_mean,back2_final_running_time,back3_final_running_time,last_final_running_time,back2_cumulative_distance_final,back3_cumulative_distance_final,last_cumulative_distance_final,back2_cumulative_strides_final,back3_cumulative_strides_final,last_cumulative_strides_final,back2_distance_efficiency_ratio,back3_distance_efficiency_ratio,last_distance_efficiency_ratio,back2_extra_distance_m,back3_extra_distance_m,last_extra_distance_m,back2_meters_per_stride,back3_meters_per_stride,last_meters_per_stride,back2_pp_race_key,back3_pp_race_key,last_pp_race_key
0,LRL|2025-12-12|1|19002714,2025-09-26,2025-09-06,2025-11-01,LRL,LRL,LRL,1.0,2.0,1.0,T,D,T,SOC,ALW,SOC,5 1/2F,1 1/16M,1M,5.5,8.5,8.0,77.0,97.0,41.0,1.5,-1.5,-1.0,1.5,1.5,1.0,0,1,0,7.0,5.0,8.0,9.0,5.0,9.0,6.0,3.0,8.0,31950.0,49000.0,31950.0,11.0,17.0,16.0,6.750000,3.333333,1.666667,6.750000,5.000000,3.500000,5.0,3.0,1.0,8.0,5.0,8.0,7.090909,4.294118,2.375000,0.943880,0.848875,1.784190,0.000000,-1.666667,-1.833333,5.850909,6.651765,6.238750,0.490111,0.638683,0.323045,5.815000,6.365000,6.111667,5.970000,7.325000,6.370000,0.155000,0.960000,0.258333,10.175000,2.866667,1.050000,24.650000,52.450000,10.333333,17.172727,22.194118,4.412500,26.7,89.4,28.2,-14.475000,-49.583333,-9.283333,0.994182,1.512471,0.272000,64.376,113.062,99.833,1107.1,1715.4,1612.3,159.1,262.8,237.3,1.000611,1.003200,1.001837,0.676,5.472,2.956,6.958517,6.527397,6.794353,19002714|LRL|2025-09-26|1,19002714|LRL|2025-09-06|2,19002714|LRL|2025-11-01|1
1,LRL|2025-12-12|1|21003278,2025-09-19,2025-04-27,2025-11-14,LRL,LRL,LRL,7.0,7.0,9.0,D,D,D,ALW,ALW,ALW,6F,1M,6F,6.0,8.0,6.0,84.0,229.0,28.0,1.0,-1.

GPS sequence feature shape: (926, 124)


### 5b. Build summary GPS features

### Why these GPS summary features?

The GPS summaries were chosen to turn detailed tracking data into a few interpretable groups of signals.

- **Basic form:** finish summaries, win rate, top-3 rate, and recency mirror the traditional PP summaries so the two feature families remain comparable.
- **Positional behavior:** early/late position and position-gain features capture whether a horse tends to lead, stalk, or finish strongly.
- **Behind-the-leader behavior:** distance-behind summaries help separate a horse that was never competitive from one that closed meaningful ground.
- **Pace and energy profile:** sectional-time summaries provide a rough description of whether the horse expended effort early or finished more efficiently late.
- **Path / efficiency:** distance-efficiency ratio and extra-distance measures capture whether the horse ran wider or covered more ground than the nominal race distance would suggest.
- **Stride-based movement:** meters per stride and cumulative distance/stride totals offer a compact way to reflect movement mechanics without keeping the full gate-by-gate series.

I included summaries here because the raw GPS signal is extremely granular. Without aggregation, it would be too easy to overfit a small dataset. These features try to keep the most interpretable race-shape information while reducing dimensionality.


In [36]:
gps_summary_features = (
    gps_map.groupby("horse_race_key")
    .agg(
        gps_num_historical_gps_starts=("pp_race_key", "nunique"),
        gps_days_since_last_gps_start=("days_since_gps_pp", "min"),
        gps_finish_mean=("official_position", "mean"),
        gps_finish_best=("official_position", "min"),
        gps_finish_std=("official_position", "std"),
        gps_win_rate=("won_gps_pp", "mean"),
        gps_top3_rate=("top3_gps_pp", "mean"),
        gps_same_surface_share=("same_surface_as_target", "mean"),
        gps_abs_distance_gap_mean=("abs_distance_gap_furlongs", "mean"),
        gps_num_gate_rows_mean=("num_gate_rows", "mean"),
        gps_early_position_mean=("early_position", "mean"),
        gps_late_position_mean=("late_position", "mean"),
        gps_position_gain_mean=("position_gain_early_to_late", "mean"),
        gps_position_gain_max=("position_gain_early_to_late", "max"),
        gps_distance_behind_early_mean=("distance_behind_early", "mean"),
        gps_distance_behind_late_mean=("distance_behind_late", "mean"),
        gps_distance_behind_recovery_mean=("distance_behind_recovery", "mean"),
        gps_distance_behind_recovery_max=("distance_behind_recovery", "max"),
        gps_sectional_early_mean=("sectional_time_early_mean", "mean"),
        gps_sectional_late_mean=("sectional_time_late_mean", "mean"),
        gps_late_minus_early_sectional_mean=("late_minus_early_sectional", "mean"),
        gps_distance_efficiency_ratio_mean=("distance_efficiency_ratio", "mean"),
        gps_distance_efficiency_ratio_min=("distance_efficiency_ratio", "min"),
        gps_distance_efficiency_ratio_max=("distance_efficiency_ratio", "max"),
        gps_extra_distance_m_mean=("extra_distance_m", "mean"),
        gps_extra_distance_m_min=("extra_distance_m", "min"),
        gps_extra_distance_m_max=("extra_distance_m", "max"),
        gps_meters_per_stride_mean=("meters_per_stride", "mean"),
        gps_meters_per_stride_min=("meters_per_stride", "min"),
        gps_meters_per_stride_max=("meters_per_stride", "max"),
        gps_final_running_time_mean=("final_running_time", "mean"),
    )
    .reset_index()
)

gps_feature_table = gps_summary_features.merge(
    gps_sequence_features,
    on="horse_race_key",
    how="left",
)

display(gps_feature_table.head(10))
print("GPS feature table shape:", gps_feature_table.shape)

,horse_race_key,gps_num_historical_gps_starts,gps_days_since_last_gps_start,gps_finish_mean,gps_finish_best,gps_finish_std,gps_win_rate,gps_top3_rate,gps_same_surface_share,gps_abs_distance_gap_mean,gps_num_gate_rows_mean,gps_early_position_mean,gps_late_position_mean,gps_position_gain_mean,gps_position_gain_max,gps_distance_behind_early_mean,gps_distance_behind_late_mean,gps_distance_behind_recovery_mean,gps_distance_behind_recovery_max,gps_sectional_early_mean,gps_sectional_late_mean,gps_late_minus_early_sectional_mean,gps_distance_efficiency_ratio_mean,gps_distance_efficiency_ratio_min,gps_distance_efficiency_ratio_max,gps_extra_distance_m_mean,gps_extra_distance_m_min,gps_extra_distance_m_max,gps_meters_per_stride_mean,gps_meters_per_stride_min,gps_meters_per_stride_max,gps_final_running_time_mean,back2_pp_race_date,back3_pp_race_date,last_pp_race_date,back2_pp_track,back3_pp_track,last_pp_track,back2_pp_race_number,back3_pp_race_number,last_pp_race_number,back2_surface,back3_surface,last_surface,back2_race_type,back3_race_type,last_race_type,back2_distance_label,back3_distance_label,last_distance_label,back2_distance_furlongs,back3_distance_furlongs,last_distance_furlongs,back2_days_since_gps_pp,back3_days_since_gps_pp,last_days_since_gps_pp,back2_distance_gap_furlongs,back3_distance_gap_furlongs,last_distance_gap_furlongs,back2_abs_distance_gap_furlongs,back3_abs_distance_gap_furlongs,last_abs_distance_gap_furlongs,back2_same_surface_as_target,back3_same_surface_as_target,last_same_surface_as_target,back2_official_position,back3_official_position,last_official_position,back2_field_size,back3_field_size,last_field_size,back2_post_position,back3_post_position,last_post_position,back2_purse,back3_purse,last_purse,back2_num_gate_rows,back3_num_gate_rows,last_num_gate_rows,back2_early_position,back3_early_position,last_early_position,back2_late_position,back3_late_position,last_late_position,back2_best_position,back3_best_position,last_best_position,back2_worst_position,back3_worst_position,last_worst_position,back2_mean_position,back3_mean_position,last_mean_position,back2_std_position,back3_std_position,last_std_position,back2_position_gain_early_to_late,back3_position_gain_early_to_late,last_position_gain_early_to_late,back2_sectional_time_mean,back3_sectional_time_mean,last_sectional_time_mean,back2_sectional_time_std,back3_sectional_time_std,last_sectional_time_std,back2_sectional_time_early_mean,back3_sectional_time_early_mean,last_sectional_time_early_mean,back2_sectional_time_late_mean,back3_sectional_time_late_mean,last_sectional_time_late_mean,back2_late_minus_early_sectional,back3_late_minus_early_sectional,last_late_minus_early_sectional,back2_distance_behind_early,back3_distance_behind_early,last_distance_behind_early,back2_distance_behind_late,back3_distance_behind_late,last_distance_behind_late,back2_distance_behind_mean,back3_distance_behind_mean,last_distance_behind_mean,back2_distance_behind_max,back3_distance_behind_max,last_distance_behind_max,back2_distance_behind_recovery,back3_distance_behind_recovery,last_distance_behind_recovery,back2_time_behind_mean,back3_time_behind_mean,last_time_behind_mean,back2_final_running_time,back3_final_running_time,last_final_running_time,back2_cumulative_distance_final,back3_cumulative_distance_final,last_cumulative_distance_final,back2_cumulative_strides_final,back3_cumulative_strides_final,last_cumulative_strides_final,back2_distance_efficiency_ratio,back3_distance_efficiency_ratio,last_distance_efficiency_ratio,back2_extra_distance_m,back3_extra_distance_m,last_extra_distance_m,back2_meters_per_stride,back3_meters_per_stride,last_meters_per_stride,back2_pp_race_key,back3_pp_race_key,last_pp_race_key
0,LRL|2025-12-12|1|19002714,3,41,6.666667,5.0,1.527525,0.0,0.0,0.333333,1.333333,14.666667,3.916667,5.083333,-1.166667,0.000000,4.697222,29.144444,-24.447222,-9.283333,6.097222,6.555000,0.457778,1.001883,1.000611,1.003200,3.034667,0.676,5.472,6.760089,6.527397,6.958517,92.

GPS feature table shape: (926, 155)


## 6. Build the final modeling feature table

This table is still **pre-modeling**.  
It contains:

- target/base context
- target variables
- traditional historical features
- historical GPS features

This is the table Notebook 3 should read.

### Why combine target, traditional, and GPS features in one final table?

The final table is designed to support the main comparison in Notebook 3:

- traditional-only models,
- GPS-only models,
- and combined models.

Putting everything into one aligned horse-race level table makes that comparison much easier and less error-prone. Each row corresponds to the same prediction unit, and each feature family can be selected or excluded cleanly.

This layout also supports more careful analysis later. For example, if GPS improves performance, you will be able to ask whether the gain comes from pace-shape information, efficiency/path information, or simply from covering horses that already look strong in the traditional features.


In [38]:
modeling_feature_table = (
    target_base
    .merge(traditional_feature_table, on="horse_race_key", how="left")
    .merge(gps_feature_table, on="horse_race_key", how="left")
)

coverage_summary = pd.DataFrame(
    {
        "metric": [
            "target horse-races",
            "horse-races with traditional history",
            "horse-races with historical GPS features",
        ],
        "value": [
            modeling_feature_table["horse_race_key"].nunique(),
            modeling_feature_table["pp_num_prior_starts"].notna().sum(),
            modeling_feature_table["gps_num_historical_gps_starts"].notna().sum(),
        ],
    }
)

display(coverage_summary)

,metric,value
0,target horse-races,1054
1,horse-races with traditional history,1013
2,horse-races with historical GPS features,926


In [39]:
display(modeling_feature_table.head(10))
print("Modeling feature table shape:", modeling_feature_table.shape)

,horse_race_key,race_key,registration_number,horse_name,target_race_date,race_number,target_distance_furlongs,distance_label,target_surface,target_race_type,target_purse,field_size,post_position,official_position,finish_percentile,is_winner,is_top_3,pp_num_prior_starts,pp_days_since_last_start,pp_finish_mean,pp_finish_best,pp_finish_worst,pp_finish_std,pp_win_rate,pp_top3_rate,pp_post_time_odds_mean,pp_post_time_odds_min,pp_field_size_mean,pp_purse_mean,pp_purse_max,pp_same_surface_share,pp_abs_distance_gap_mean,pp_abs_distance_gap_min,pp_has_gps_history_share,pp_num_active_pocs_mean,pp_early_call_position_mean,pp_late_call_position_mean,pp_early_to_late_gain_mean,pp_early_to_finish_gain_mean,pp_early_length_behind_mean,pp_late_length_behind_mean,pp_finish_length_behind_mean,pp_length_recovery_mean,back2_pp_race_date_x,back3_pp_race_date_x,last_pp_race_date_x,back2_pp_track_x,back3_pp_track_x,last_pp_track_x,back2_pp_race_number_x,back3_pp_race_number_x,last_pp_race_number_x,back2_surface_x,back3_surface_x,last_surface_x,back2_race_type_x,back3_race_type_x,last_race_type_x,back2_distance_label_x,back3_distance_label_x,last_distance_label_x,back2_distance_furlongs_x,back3_distance_furlongs_x,last_distance_furlongs_x,back2_days_since_pp,back3_days_since_pp,last_days_since_pp,back2_distance_gap_furlongs_x,back3_distance_gap_furlongs_x,last_distance_gap_furlongs_x,back2_abs_distance_gap_furlongs_x,back3_abs_distance_gap_furlongs_x,last_abs_distance_gap_furlongs_x,back2_same_surface_as_target_x,back3_same_surface_as_target_x,last_same_surface_as_target_x,back2_official_position_x,back3_official_position_x,last_official_position_x,back2_field_size_x,back3_field_size_x,last_field_size_x,back2_post_position_x,back3_post_position_x,last_post_position_x,back2_post_time_odds,back3_post_time_odds,last_post_time_odds,back2_purse_x,back3_purse_x,last_purse_x,back2_is_about_distance,back3_is_about_distance,last_is_about_distance,back2_is_graded_race,back3_is_graded_race,last_is_graded_race,back2_has_gps_data,back3_has_gps_data,last_has_gps_data,back2_n_active_point_of_calls,back3_n_active_point_of_calls,last_n_active_point_of_calls,back2_early_call_position,back3_early_call_position,last_early_call_position,back2_mid_call_position,back3_mid_call_position,last_mid_call_position,back2_late_call_position,back3_late_call_position,last_late_call_position,back2_early_call_length_behind,back3_early_call_length_behind,last_early_call_length_behind,back2_late_call_length_behind,back3_late_call_length_behind,last_late_call_length_behind,back2_finish_length_behind,back3_finish_length_behind,last_finish_length_behind,back2_early_to_late_call_gain,back3_early_to_late_call_gain,last_early_to_late_call_gain,back2_early_to_finish_gain,...,gps_meters_per_stride_max,gps_final_running_time_mean,back2_pp_race_date_y,back3_pp_race_date_y,last_pp_race_date_y,back2_pp_track_y,back3_pp_track_y,last_pp_track_y,back2_pp_race_number_y,back3_pp_race_number_y,last_pp_race_number_y,back2_surface_y,back3_surface_y,last_surface_y,back2_race_type_y,back3_race_type_y,last_race_type_y,back2_distance_label_y,back3_distance_label_y,last_distance_label_y,back2_distance_furlongs_y,back3_distance_furlongs_y,last_distance_furlongs_y,back2_days_since_gps_pp,back3_days_since_gps_pp,last_days_since_gps_pp,back2_distance_gap_furlongs_y,back3_distance_gap_furlongs_y,last_distance_gap_furlongs_y,back2_abs_distance_gap_furlongs_y,back3_abs_distance_gap_furlongs_y,last_abs_distance_gap_furlongs_y,back2_same_surface_as_target_y,back3_same_surface_as_target_y,last_same_surface_as_target_y,back2_official_position_y,back3_official_position_y,last_official_position_y,back2_field_size_y,back3_field_size_y,last_field_size_y,back2_post_position_y,back3_post_position_y,last_post_position_y,back2_purse_y,back3_purse_y,last_purse_y,back2_num_gate_rows,back3_num_gate_rows,last_num_gate_rows,back2_early_position,back3_early_position,last_early_position,back2_late_position,back3_late_position,last_late_

Modeling feature table shape: (1054, 287)


## 7. Build a feature inventory

This makes it easier to keep Notebook 4 organized.

### Why create a feature inventory?

A feature inventory may feel administrative, but it is actually useful for modeling discipline.

It gives a clear record of which columns belong to which feature family, which helps with:

- building traditional-only, GPS-only, and combined design matrices,
- checking that no target or leakage columns accidentally enter training,
- summarizing how wide each feature block is,
- and interpreting results later by feature group rather than by one long unstructured list of columns.

This becomes especially helpful once you start running multiple models and ablations in Notebook 3.


In [41]:
target_columns = [
    "horse_race_key",
    "race_key",
    "registration_number",
    "horse_name",
    "target_race_date",
    "race_number",
    "target_distance_furlongs",
    "distance_label",
    "target_surface",
    "target_race_type",
    "target_purse",
    "field_size",
    "post_position",
    "official_position",
    "finish_percentile",
    "is_winner",
    "is_top_3",
]

traditional_summary_cols = [c for c in traditional_summary_features.columns if c != "horse_race_key"]
traditional_sequence_cols = [c for c in pp_sequence_features.columns if c != "horse_race_key"]
gps_summary_cols = [c for c in gps_summary_features.columns if c != "horse_race_key"]
gps_sequence_cols_final = [c for c in gps_sequence_features.columns if c != "horse_race_key"]

feature_groups = {
    "target_and_outcomes": target_columns,
    "traditional_summary_features": traditional_summary_cols,
    "traditional_sequence_features": traditional_sequence_cols,
    "gps_summary_features": gps_summary_cols,
    "gps_sequence_features": gps_sequence_cols_final,
}

feature_inventory = pd.DataFrame(
    [(group, feature) for group, feats in feature_groups.items() for feature in feats],
    columns=["feature_group", "feature_name"],
)

group_sizes = (
    feature_inventory.groupby("feature_group")
    .size()
    .rename("n_features")
    .reset_index()
    .sort_values("feature_group")
)

display(group_sizes)

,feature_group,n_features
0,gps_sequence_features,123
1,gps_summary_features,31
2,target_and_outcomes,17
3,traditional_sequence_features,90
4,traditional_summary_features,26


In [43]:
display(feature_inventory.head(40))

,feature_group,feature_name
0,target_and_outcomes,horse_race_key
1,target_and_outcomes,race_key
2,target_and_outcomes,registration_number
3,target_and_outcomes,horse_name
4,target_and_outcomes,target_race_date
5,target_and_outcomes,race_number
6,target_and_outcomes,target_distance_furlongs
7,target_and_outcomes,distance_label
8,target_and_outcomes,target_surface
9,target_and_outcomes,target_race_type


## 8. Save feature outputs

Notebook 4 should read `modeling_feature_table.csv` plus the feature inventory.

In [46]:
target_base_path = FEATURE_DIR / "target_base_table.csv"
historical_pp_selected_path = FEATURE_DIR / "historical_pp_selected.csv"
traditional_feature_table_path = FEATURE_DIR / "traditional_feature_table.csv"
gps_pp_race_summary_path = FEATURE_DIR / "gps_pp_race_summary.csv"
historical_gps_selected_path = FEATURE_DIR / "historical_gps_selected.csv"
gps_feature_table_path = FEATURE_DIR / "gps_feature_table.csv"
modeling_feature_table_path = FEATURE_DIR / "modeling_feature_table.csv"
feature_inventory_path = FEATURE_DIR / "feature_inventory.csv"
feature_groups_json_path = FEATURE_DIR / "feature_groups.json"

target_base.to_csv(target_base_path, index=False)
pp_map.to_csv(historical_pp_selected_path, index=False)
traditional_feature_table.to_csv(traditional_feature_table_path, index=False)
gps_pp_race_summary.to_csv(gps_pp_race_summary_path, index=False)
gps_map.to_csv(historical_gps_selected_path, index=False)
gps_feature_table.to_csv(gps_feature_table_path, index=False)
modeling_feature_table.to_csv(modeling_feature_table_path, index=False)
feature_inventory.to_csv(feature_inventory_path, index=False)

with open(feature_groups_json_path, "w", encoding="utf-8") as f:
    json.dump(feature_groups, f, indent=2)

saved_outputs = pd.DataFrame(
    {
        "file": [
            target_base_path.name,
            historical_pp_selected_path.name,
            traditional_feature_table_path.name,
            gps_pp_race_summary_path.name,
            historical_gps_selected_path.name,
            gps_feature_table_path.name,
            modeling_feature_table_path.name,
            feature_inventory_path.name,
            feature_groups_json_path.name,
        ],
        "path": [
            str(target_base_path),
            str(historical_pp_selected_path),
            str(traditional_feature_table_path),
            str(gps_pp_race_summary_path),
            str(historical_gps_selected_path),
            str(gps_feature_table_path),
            str(modeling_feature_table_path),
            str(feature_inventory_path),
            str(feature_groups_json_path),
        ],
    }
)
display(saved_outputs)

,file,path
0,target_base_table.csv,../data/features/target_base_table.csv
1,historical_pp_selected.csv,../data/features/historical_pp_selected.csv
2,traditional_feature_table.csv,../data/features/traditional_feature_table.csv
3,gps_pp_race_summary.csv,../data/features/gps_pp_race_summary.csv
4,historical_gps_selected.csv,../data/features/historical_gps_selected.csv
5,gps_feature_table.csv,../data/features/gps_feature_table.csv
6,modeling_feature_table.csv,../data/features/modeling_feature_table.csv
7,feature_inventory.csv,../data/features/feature_inventory.csv
8,feature_groups.json,../data/features/feature_groups.json


## 9. Final checkpoint

At the end of Notebook 3, we now have:

- a clean target/base table
- a mapped historical traditional PP table
- a mapped historical GPS table
- a traditional feature table
- a GPS feature table
- one combined modeling feature table
- a feature inventory that lists what each group contains

That is the right handoff point for Notebook 4, where we can do:

- train / validation splits
- traditional-only vs GPS-only vs combined model comparison
- performance metrics
- feature importance / interpretation